# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 5: *Feature Interaction Analysis*
##### Version Number: 2.0
---
### Contents  
> 1. *Load Data*
> 2. *Split and Scale Data*
> 3. *Feature Interaction Analysis*
> 4. *Export Data*
---
### Notes
---
### Inputs
- `final.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_3rd_degree_interactions

# A space saving function to rank interactions
from src.data_utils import rank_interactions_by_correlation

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize

# Modeling prep
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import PolynomialFeatures

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 1. Load Data

In [3]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/trimmed.csv")
details = pd.read_csv("../data/processed/details.csv")

In [4]:
final

,Sample_ID,Date,Sample_Elevation,Region_ID,Interface_Area,Intermix_Area,Influence_Area,ETo (in),Precip (in),Sol Rad (Ly/day),...,Mean Income,Target,Days Without Rain,2-Year Avg Fires,Avg Air Temp (F) 7 Day Avg,Precip (in) 7 Day Avg,Avg Rel Hum (%) 7 Day Avg,Avg Wind Speed (mph) 7 Day Avg,Month,Year
0,1,2018-01-01,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.05,0.02,230.0,...,111241,0,0,97.630952,55.400000,0.020000,96.000000,3.100000,1,2018
1,1,2018-01-02,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.00,279.0,...,111241,0,1,97.630952,56.950000,0.010000,93.000000,3.300000,1,2018
2,1,2018-01-03,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.02,0.02,125.0,...,111241,0,0,97.630952,57.900000,0.013333,92.333333,3.166667,1,2018
3,1,2018-01-04,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.01,268.0,...,111241,0,0,97.630952,58.350000,0.012500,88.750000,3.275000,1,2018
4,1,2018-01-05,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.00,255.0,...,111241,0,1,97.630952,58.620000,0.010000,88.000000,3.520000,1,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,173,2025-01-19,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.04,0.00,252.0,...,59138,0,13,30.500000,29.957143,0.000000,73.285714,2.885714,1,2025
447776,173,2025-01-20,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.04,0.00,266.0,...,59138,0,14,30.500000,29.814286,0.000000,71.142857,2.971429,1,2025
447777,173,2025-01-21,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.05,0.00,250.0,...,59138,0,15,30.500000,29.542857,0.000000,68.428571,3.085714,1,2025
447778,173,2025-01-22,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.05,0.00,267.0,...,59138,0,16,30.500000,28.742857,0.000000,66.285714,3.128571,1,2025


---

## 2. Split and Scale Data

In [5]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target']

y = final['Target']
X = final.drop(columns=text_columns)
#details = final[text_columns]

In [6]:
X

,Sample_Elevation,Region_ID,Interface_Area,Intermix_Area,Influence_Area,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Avg Air Temp (F),...,density,Mean Income,Days Without Rain,2-Year Avg Fires,Avg Air Temp (F) 7 Day Avg,Precip (in) 7 Day Avg,Avg Rel Hum (%) 7 Day Avg,Avg Wind Speed (mph) 7 Day Avg,Month,Year
0,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.05,0.02,230.0,14.3,55.4,...,777.337917,111241,0,97.630952,55.400000,0.020000,96.000000,3.100000,1,2018
1,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.00,279.0,15.1,58.5,...,777.337917,111241,1,97.630952,56.950000,0.010000,93.000000,3.300000,1,2018
2,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.02,0.02,125.0,16.0,59.8,...,777.337917,111241,0,97.630952,57.900000,0.013333,92.333333,3.166667,1,2018
3,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.01,268.0,13.7,59.7,...,777.337917,111241,0,97.630952,58.350000,0.012500,88.750000,3.275000,1,2018
4,36.609302,7,4.037876e+08,1.709527e+08,5.663377e+08,0.06,0.00,255.0,14.8,59.7,...,777.337917,111241,1,97.630952,58.620000,0.010000,88.000000,3.520000,1,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.04,0.00,252.0,3.5,28.1,...,2.169602,59138,13,30.500000,29.957143,0.000000,73.285714,2.885714,1,2025
447776,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.04,0.00,266.0,2.9,26.5,...,2.169602,59138,14,30.500000,29.814286,0.000000,71.142857,2.971429,1,2025
447777,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.05,0.00,250.0,2.9,27.9,...,2.169602,59138,15,30.500000,29.542857,0.000000,68.428571,3.085714,1,2025
447778,1618.552856,1,1.194300e+06,6.072592e+06,1.144408e+08,0.05,0.00,267.0,3.1,29.2,...,2.169602,59138,16,30.500000,28.742857,0.000000,66.285714,3.128571,1,2025


Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [7]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

X_scaled = X

---

## 3. Interaction Correlation analysis

For all independent variables create a new dataframe containing every possible interaction of 2nd degree

In [8]:
inter_X = create_2nd_degree_interactions(X_scaled)

inter_X  = create_3rd_degree_interactions(X_scaled)

Combine interaction terms with single variables for correlation analysis

In [9]:
inter_X_combined = pd.concat([X_scaled, inter_X], axis=1)

In [10]:
correlation_results = rank_interactions_by_correlation(inter_X_combined, y)

C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [11]:
correlation_results.head(20)

,Feature,Correlation
0,ETo (in) x Season,0.128950
1,ETo (in) x Month,0.121300
2,Sol Rad (Ly/day) x Season,0.115567
3,ETo (in) x Avg Vap Pres (mBars),0.112727
4,ETo (in) x 2-Year Avg Fires,0.112472
5,ETo (in) x Avg Soil Temp (F),0.111070
6,ETo (in) x Avg Air Temp (F) 7 Day Avg,0.109585
7,ETo (in) x Avg Air Temp (F),0.108448
8,Sol Rad (Ly/day) x Month,0.105817
9,Sol Rad (Ly/day) x 2-Year Avg Fires,0.105164


In [12]:
keep = correlation_results.head(40)['Feature'].tolist()
#keep = correlation_results['Feature'].tolist()
top_20 = inter_X_combined[keep]
top_20

,ETo (in) x Season,ETo (in) x Month,Sol Rad (Ly/day) x Season,ETo (in) x Avg Vap Pres (mBars),ETo (in) x 2-Year Avg Fires,ETo (in) x Avg Soil Temp (F),ETo (in) x Avg Air Temp (F) 7 Day Avg,ETo (in) x Avg Air Temp (F),Sol Rad (Ly/day) x Month,Sol Rad (Ly/day) x 2-Year Avg Fires,...,Avg Vap Pres (mBars) x Season,Avg Vap Pres (mBars) x Avg Air Temp (F),Avg Wind Speed (mph) x 2-Year Avg Fires,Avg Vap Pres (mBars) x Avg Soil Temp (F),Wind Run (miles) x 2-Year Avg Fires,ETo (in) x Avg Rel Hum (%) 7 Day Avg,2-Year Avg Fires,Influence_Area x ETo (in),Avg Soil Temp (F) x Month,Influence_Area x 2-Year Avg Fires
0,0.0,0.0,0.0,0.027666,0.016383,0.032513,0.044370,0.048861,0.0,0.005922,...,0.0,0.146209,0.017607,0.097291,0.011295,0.092308,0.170386,0.026156,0.0,0.046348
1,0.0,0.0,0.0,0.035056,0.019660,0.040528,0.055055,0.061873,0.0,0.007183,...,0.0,0.162920,0.019878,0.106716,0.012719,0.107308,0.170386,0.031387,0.0,0.046348
2,0.0,0.0,0.0,0.012382,0.006553,0.013761,0.018722,0.021077,0.0,0.003218,...,0.0,0.176421,0.016471,0.115187,0.010644,0.035513,0.170386,0.010462,0.0,0.046348
3,0.0,0.0,0.0,0.031806,0.019660,0.042343,0.056691,0.063127,0.0,0.006900,...,0.0,0.150811,0.020446,0.101157,0.012991,0.102404,0.170386,0.031387,0.0,0.046348
4,0.0,0.0,0.0,0.034360,0.019660,0.042041,0.057006,0.063127,0.0,0.006565,...,0.0,0.162920,0.025558,0.108499,0.016353,0.101538,0.170386,0.031387,0.0,0.046348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,0.0,0.0,0.0,0.005417,0.004095,0.002722,0.015678,0.020067,0.0,0.002027,...,0.0,0.018371,0.006565,0.002492,0.004163,0.056374,0.053229,0.004228,0.0,0.002926
447776,0.0,0.0,0.0,0.004488,0.004095,0.002621,0.015567,0.018952,0.0,0.002139,...,0.0,0.014376,0.006387,0.001988,0.004044,0.054725,0.053229,0.004228,0.0,0.002926
447777,0.0,0.0,0.0,0.005611,0.005118,0.003024,0.019195,0.024909,0.0,0.002011,...,0.0,0.015116,0.005323,0.001835,0.003358,0.065797,0.053229,0.005285,0.0,0.002926
447778,0.0,0.0,0.0,0.005998,0.005118,0.002772,0.018416,0.026042,0.0,0.002147,...,0.0,0.016893,0.005500,0.001798,0.003486,0.063736,0.053229,0.005285,0.0,0.002926


Keep top 20 results

In [13]:
# Recompute correlation matrix
corr_matrix = top_20.corr()

to_drop = set()
cols = corr_matrix.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if corr_matrix.iloc[i, j] > 0.9:
            col_to_drop = cols[j]  # drop the later column in the pair
            to_drop.add(col_to_drop)

df_reduced = top_20.drop(columns=list(to_drop))

df_reduced

,ETo (in) x Season,ETo (in) x Avg Vap Pres (mBars),ETo (in) x 2-Year Avg Fires,ETo (in) x Avg Soil Temp (F),Avg Air Temp (F) 7 Day Avg,Season x Avg Air Temp (F) 7 Day Avg,Avg Vap Pres (mBars) x Avg Air Temp (F) 7 Day Avg,Avg Vap Pres (mBars) x Season,Avg Wind Speed (mph) x 2-Year Avg Fires,ETo (in) x Avg Rel Hum (%) 7 Day Avg,Influence_Area x ETo (in),Avg Soil Temp (F) x Month,Influence_Area x 2-Year Avg Fires
0,0.0,0.027666,0.016383,0.032513,0.461449,0.0,0.132771,0.0,0.017607,0.092308,0.026156,0.0,0.046348
1,0.0,0.035056,0.019660,0.040528,0.477145,0.0,0.144967,0.0,0.019878,0.107308,0.031387,0.0,0.046348
2,0.0,0.012382,0.006553,0.013761,0.486764,0.0,0.156705,0.0,0.016471,0.035513,0.010462,0.0,0.046348
3,0.0,0.031806,0.019660,0.042343,0.491321,0.0,0.135434,0.0,0.020446,0.102404,0.031387,0.0,0.046348
4,0.0,0.034360,0.019660,0.042041,0.494055,0.0,0.147123,0.0,0.025558,0.101538,0.031387,0.0,0.046348
...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,0.0,0.005417,0.004095,0.002722,0.203819,0.0,0.014353,0.0,0.006565,0.056374,0.004228,0.0,0.002926
447776,0.0,0.004488,0.004095,0.002621,0.202372,0.0,0.011808,0.0,0.006387,0.054725,0.004228,0.0,0.002926
447777,0.0,0.005611,0.005118,0.003024,0.199624,0.0,0.011648,0.0,0.005323,0.065797,0.005285,0.0,0.002926
447778,0.0,0.005998,0.005118,0.002772,0.191523,0.0,0.011946,0.0,0.005500,0.063736,0.005285,0.0,0.002926


In [14]:
X_reduced = df_reduced

---

In [15]:
final['Date'] = pd.to_datetime(final['Date'])

# Filter target 0 rows **before 2025-01-01**
target_0_pre2025_idx = final[
    (final['Target'] == 0) &
    (final['Date'] < '2025-01-01')
].index

# Sample from these pre-2025 target 0 rows
target_0_sampled_idx = target_0_pre2025_idx.to_series().sample(n=10000, random_state=42).index

# Keep all target 1 and 2 rows
target_1_2_idx = final[final['Target'].isin([1, 2])].index

# Keep target 0 rows **on or after 2025-01-01**
target_0_post2025_idx = final[
    (final['Target'] == 0) &
    (final['Date'] >= '2025-01-01')
].index

# Combine all indexes
indexes_to_keep = target_0_sampled_idx.union(target_0_post2025_idx).union(target_1_2_idx)

# Subset your datasets
X_reduced_2 = X_reduced.loc[indexes_to_keep]
y_reduced = y.loc[indexes_to_keep]
details_reduced = details.loc[indexes_to_keep]

# Check class counts
print(y_reduced.value_counts())

Target
0    13944
2     4772
1     3429
Name: count, dtype: int64


## 4. Export Data

In [16]:
X_reduced_2.to_csv('../data/processed/X_reduced.csv', index=False)
y_reduced.to_csv('../data/processed/y_reduced.csv', index=False)
details_reduced.to_csv('../data/processed/details_reduced.csv', index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
